In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import os

In [5]:
# Duomenų rinkiniai ir klasterių skaičiai pagal:
# Empirinis, Alkūnės, Vidutinio silueto metodus
cluster_counts = {
    "norm_full.csv": [76, 6, 3, 9],
    "norm_6.csv": [27, 7, 13, 8],
    "dvi_dim.csv": [27, 10, 11, 10]
}
cluster_labels = ["empirinis", "alkunes", "silueto", "dendograma"]

base_folder = "hierarchical"  # Pagrindinis aplankas rezultatams
linkage_method = "ward"       # Naudojamas sujungimo metodas
metric_type = "euclidean"     # Naudojama metrika


In [6]:
for file in cluster_counts.keys():
    print(f"\nApdorojamas duomenų failas: {file}")
    df = pd.read_csv(file)

    if 'label' in df.columns:
        X = df.drop(columns=['label']).values
    else:
        X = df.values

    print(f"Duomenys nuskaityti: {file}")

    # Sukuriam aplankus rezultatams išsaugoti
    file_folder_orig = os.path.join(base_folder, "original", file[:-4])
    file_folder_clean = os.path.join(base_folder, "cleaned", file[:-4])
    os.makedirs(file_folder_orig, exist_ok=True)
    os.makedirs(file_folder_clean, exist_ok=True)

    # t-SNE
    if file == "dvi_dim.csv":
        X_tsne = np.array(X)
    else:
        if file == "norm_full.csv":
            perp, learn_r, early_ex = 50, 200, 24
        elif file == "norm_6.csv":
            perp, learn_r, early_ex = 50, 50, 12

        tsne = TSNE(
            n_components=2,         # sumažiname duomenų dimensijų skaičių iki 2
            perplexity=perp,          # kiek artimiausių kaimynų laikoma reikšmingais
            learning_rate=learn_r,      # nustato, kokio dydžio korekcijos taikomos kiekviename iteracijos žingsnyje
            max_iter=1000,            # iteracijų skaičius: kiek gradientinio nusileidimo žingsnių atliekama minimizuojant klaidą
            early_exaggeration=early_ex,# padeda atskirti klasterius ir išvengti lokalių minimumų
            metric="euclidean",     # atstumo metrika pradinėje erdvėje
            random_state=67,        # kad rezultatai būtų atkuriami (užtikrina tą pačią pradinę taškų padėtį)
            init="pca"              # pradinė taškų pozicija nustatoma PCA pagrindu (stabiliau nei random)
        )
        X_tsne = tsne.fit_transform(X)

    # Dendogramos kurimas
    # print("Kuriama dendrograma (tik originaliems duomenims)...")
    # linked = linkage(X, method=linkage_method, metric=metric_type)
    # plt.figure(figsize=(12, 6))
    # dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=False)
    # plt.title(f"{file} — Hierarchinio klasterizavimo dendrograma\n({linkage_method.capitalize()} + {metric_type.capitalize()})")
    # plt.xlabel("Duomenų taškai")
    # plt.ylabel("Atstumas")
    # plt.tight_layout()
    # plt.savefig(os.path.join(file_folder_orig, f"dendrogram_{file[:-4]}.png"), dpi=200)
    # plt.close()

    # Klasterizavimas ir silueto koeficiento skaičiavimas tiek originaliems, tiek duomenims su pašalintomis išskirtimis

    # IQR metodas išskirtims pašalinti
    Q1 = np.percentile(X, 10, axis=0)
    Q3 = np.percentile(X, 90, axis=0)
    IQR = Q3 - Q1

    # Naudojame k = 3.0, kad pašalintume tik ekstremalias išskirtis, palikdami salygines isskirtis
    k = 3.0
    mask = ~((X < (Q1 - k * IQR)) | (X > (Q3 + k * IQR))).any(axis=1)
    X_clean = X[mask]

    removed = np.sum(~mask)
    total = len(mask)
    print(f"Išskirtys pašalintos: {removed} iš {total} duomenų taškų ({removed/total:.2%})")

    datasets = {"original": X, "cleaned": X_clean}

    # Pagrindinis ciklas
    for version_name, X_used in datasets.items():
        print(f"\nVykdomas variantas: {version_name.upper()}")
        folder = file_folder_orig if version_name == "original" else file_folder_clean

        if version_name == "cleaned":
            if file == "dvi_dim.csv":
                X_tsne_used = np.array(X_used)
            else:
                tsne = TSNE(
                    n_components=2,
                    perplexity=perp,
                    learning_rate=learn_r,
                    max_iter=1000,
                    early_exaggeration=early_ex,
                    metric="euclidean",
                    random_state=67,
                    init="pca"
                )
                X_tsne_used = tsne.fit_transform(X_used)
        else:
            X_tsne_used = X_tsne

        # Klasterizavimas ir siluetu palyginimas
        for label, k in zip(cluster_labels, cluster_counts[file]):
            print(f"{label} metodas (k={k})")

            agg = AgglomerativeClustering(
                n_clusters=k,
                linkage=linkage_method,
                metric=metric_type
            )
            y_pred = agg.fit_predict(X_used)

            # Siluetu iverciai
            sil_orig = silhouette_score(X_used, y_pred)
            sil_tsne = silhouette_score(X_tsne_used, y_pred)
            print(f"Silueto koef. originalioje erdvėje: {sil_orig:.3f}")
            print(f"Silueto koef. po t-SNE: {sil_tsne:.3f}")

            with open(os.path.join(folder, "silhouette_scores.txt"), "a", encoding="utf-8") as f:
                f.write(f"{label} (k={k}): orig={sil_orig:.3f}, tsne={sil_tsne:.3f}\n")

            # Vizualizacija t-SNE
            plt.figure(figsize=(9, 9))
            cmap = ListedColormap(list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors) + list(plt.cm.tab20c.colors))
            plt.scatter(X_tsne_used[:, 0], X_tsne_used[:, 1], c=y_pred, cmap=cmap, s=50)
            plt.title(f"{file} ({version_name}) — t-SNE pagal hierarchinius klasterius\n({label}, k={k})")
            plt.xlabel("t-SNE komponentė 1")
            plt.ylabel("t-SNE komponentė 2")
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(os.path.join(folder, f"tsne_{file[:-4]}_{label}_{k}_{version_name}.png"), dpi=200)
            plt.close()

        print(f"{version_name.upper()} versijos rezultatai išsaugoti aplanke: {folder}")

print("\nVisi duomenų rinkiniai apdoroti (originalūs ir su pašalintomis išskirtimis).")


Apdorojamas duomenų failas: norm_full.csv
Duomenys nuskaityti: norm_full.csv
Išskirtys pašalintos: 555 iš 11818 duomenų taškų (4.70%)

Vykdomas variantas: ORIGINAL
empirinis metodas (k=76)
Silueto koef. originalioje erdvėje: 0.266
Silueto koef. po t-SNE: 0.355
alkunes metodas (k=6)
Silueto koef. originalioje erdvėje: 0.212
Silueto koef. po t-SNE: 0.285
silueto metodas (k=3)
Silueto koef. originalioje erdvėje: 0.297
Silueto koef. po t-SNE: 0.288
dendograma metodas (k=9)
Silueto koef. originalioje erdvėje: 0.236
Silueto koef. po t-SNE: 0.258
ORIGINAL versijos rezultatai išsaugoti aplanke: hierarchical\original\norm_full

Vykdomas variantas: CLEANED
empirinis metodas (k=76)
Silueto koef. originalioje erdvėje: 0.215
Silueto koef. po t-SNE: 0.319
alkunes metodas (k=6)
Silueto koef. originalioje erdvėje: 0.229
Silueto koef. po t-SNE: 0.243
silueto metodas (k=3)
Silueto koef. originalioje erdvėje: 0.320
Silueto koef. po t-SNE: 0.264
dendograma metodas (k=9)
Silueto koef. originalioje erdvėje